# Stage 2 Notebook 21 - Exp2P DETR-style lane queries

**The big move.** Exp2O's oracle diagnostic confirmed (NB20):

- `decoded_oracle_f1 = 0.27 - 0.32` (perfect-ranking ceiling on current geometry).
- `decoded_f1 = 0.007 - 0.015` (cls-ranked).
- The 30x gap is purely from cls failure.

Across Exp2G/H/I/J/K/L/M/N (eight cls-rescue attempts spanning focal, ASL, OHEM, separate path, IoU regression, QFL, sqrt rescaling, separate cls feature pathway), the prior-based head's cls always collapsed into the same all-priors-cluster-at-one-value attractor. The bottleneck is **the prior-based design itself**, not any specific cls loss.

Exp2P abandons the 192-prior + dynamic-k design entirely:

- **K=12 learned lane queries** (DETR/RMT-PPAD/GANet style). Each query is its own learned positional embedding.
- **3-layer transformer decoder** that cross-attends to flattened multi-scale features (with per-scale embedding so queries can attend by feature level).
- **Per-query outputs**: cls_logit (1d), curve params (4d: start_y, start_x, theta, length), row offsets (72d). Curves are computed deterministically from params + offsets, same parameterization as CLRKDLaneHead.
- **Hungarian matching** between K queries and N GT lanes by combined cost (cls_cost + point_cost + iou_cost + xytl_cost). One-to-one, deterministic per image. Matched query: cls=1 + geometry losses. Unmatched query: cls=0.
- **No NMS needed at inference**: queries are intrinsically non-redundant via the decoder's self-attention. Top-K by sigmoid score still works for evaluation consistency.

Why this works where the prior-based head failed:

- **Balanced positive rate.** ~5 GT lanes / 12 queries = 42 % positives. The prior-based head had ~5 / 192 = 2.6 %, which drove BCE/focal toward predicting all-zero.
- **No batch-to-batch matching instability.** The dynamic-k matcher reshuffles which 5-of-192 priors are positive each batch; queries are matched 1-to-1 to GT and converge to specialize.
- **Each query has a distinct learned positional embedding from initialization.** The all-cluster-at-0.13 attractor cannot form because queries are not tied to a fixed spatial anchor.

Backbone (RMT + GCA + AIFI), detection head (DETR), all eval infrastructure (Exp2N decoder + Exp2O oracle metric) are unchanged. This notebook tests *one* hypothesis: is the lane head's paradigm the bottleneck?

Reference: Carion et al. 2020 'End-to-End Object Detection with Transformers' (DETR); RMT-PPAD lane formulation.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. Do not rerun Notebook 00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp16_rmt_gca_lane_query_head_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: forward + backward through LaneQueryHead with K=12 queries.
# Must print 'OK exp16_*.yaml' with shapes lane_shape=(1, K, 72, 2). Note K
# may be the smoke test's reduced size (the smoke harness shrinks num_priors
# to 16; for query head it should still pass since K is a config knob).
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp16_rmt_gca_lane_query_head_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp16_rmt_gca_lane_query_head_joint_smoke.log
OK exp16_rmt_gca_lane_query_head_joint.yaml
  lane_shape=(1, 12, 72, 2) det_shape=(1, 4, 4)
  lane_loss=0.7553 det_loss=3.4434 grad_cos=-0.0144 lambda_lane=0.0625
  gate_stats={'gate/det_mean': 0.49886855483055115, 'gate/lane_mean': 0.5026520490646362, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp16_rmt_gca_lane_query_head_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp16_rmt_gca_lane_query_head_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp16_rmt_gca_lane_query_head_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp16_rmt_gca_lane_query_head_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp16_rmt_gca_lane_query_head_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp16_rmt_gca_lane_query_head_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp16_rmt_gca_lane_query_head_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd

0

## What to watch in Exp2P training

Reference epoch 10 across recent experiments:
- Exp2N (prior-based): `decoded_f1=0.011`, `oracle_f1` not measured.
- Exp2O (prior-based + oracle): `decoded_f1=0.008`, `oracle_f1=0.273`. The 30x gap is the diagnosis.

Strong signals that the query design fixes ranking:

- `pos_score_mean - neg_score_mean >= 0.30` at epoch 10. With 5/12 queries positive and 7/12 negative, score separation should be very clear. Compare to prior-based runs where this stayed at ~+0.001.
- `val/lane_exist_best_f1 >= 0.50` at epoch 10. 12 queries is a small enough output set that even modest learning produces meaningful F1.
- **`val/lane/decoded_f1 >= 0.10` at epoch 10**, ideally >= 0.20. CLRKDNet-comparable territory by epoch 10 would be remarkable; we're aiming to break the 0.01 floor.
- `val/lane/decoded_oracle_f1` is reported alongside decoded_f1; if both rise together, query design works AND geometry holds.
- **Geometry holds**: `val/matched_line_iou >= 0.30`, `val/lane_point_mae <= 0.40`. Looser bounds than Exp2M (0.40 / 0.34) since the query head needs more epochs to converge curve geometry from scratch.
- `train/lane/cls` strictly decreases over training (in prior-based runs cls plateaued from the start).

Failure signals -> next ablation:

- `decoded_f1 < 0.05` at epoch 10 -> queries also can't learn in 10 epochs. Two plausible causes: (a) need denoising queries or auxiliary decoder losses (RT-DETR convention) for faster convergence, (b) need 30+ epochs (queries take longer than priors to converge).
- Geometry collapses (matched_iou < 0.20) -> query parameterization can't represent BDD lanes. Switch to Bezier curve params or learned anchor offsets.
- Score separation is positive but tiny (< 0.10) and best_f1 < 0.20 -> queries are failing in the same way priors did, and the bottleneck is something deeper (data, geometry, training duration).

After short10, run NB08 to plot Exp2K / Exp2L / Exp2M / Exp2N / Exp2O / Exp2P side-by-side, with both `lane/decoded_f1` and `lane/decoded_oracle_f1` curves visible.